# AxiCLASS axion example without fluid approximation

This notebook is adapted from `example_axion.ipynb`, but it:

- disables the scalar-field fluid approximation,
- disables the axionCAMB-like evolution shortcut,
- forces Runge-Kutta evolution for both background and perturbations,
- exposes explicit background and perturbation step sizes for testing convergence.


In [ ]:
from classy import Class
import matplotlib.pyplot as plt
import numpy as np
import matplotlib

from matplotlib import rc

rc('font', **{'family': 'serif', 'serif': ['Times']})
rc('text', usetex=True)
matplotlib.mathtext.rcParams['legend.fontsize'] = 'medium'
plt.rcParams['figure.figsize'] = [8.0, 6.0]


In [ ]:
def compute_axion_nofluid(m_axion, f_axion, Omega_scf,\n                         bg_stepsize=0.05, pert_stepsize=0.05, pert_sampling_stepsize=0.05,\n                         tol_background=1e-12, tol_pert=1e-6):
    """
    Run the axion model with the full scalar-field evolution (no fluid approximation)
    and user-controlled RK step sizes in the background and perturbation solvers.
    """

    cosmofid = {
        'output': 'mPk',
        'z_max_pk': 1,
        'P_k_max_h/Mpc': 1.,
        'Omega_cdm': 0.3,
        'H0': 67,
        'Omega_scf': Omega_scf,
        'm_axion': m_axion,
        'f_axion': f_axion,
        'scf_parameters': r'%g, %g' % (0.05, 0.),
        'scf_potential': 'axion',
        'n_axion': 1,
        'scf_has_perturbations': 'yes',
        'attractor_ic_scf': 'no',
        'do_shooting': 'yes',
        'do_shooting_scf': 'yes',
        'scf_tuning_index': 0,
        'tol_shooting_deltax': 1e-4,
        'tol_shooting_deltaF': 1e-4,
        'background_verbose': 10,
        'thermodynamics_verbose': 1,

        # Disable fluid / approximate evolution
        'scf_evolve_as_fluid': 'no',
        'scf_evolve_like_axionCAMB': 'no',

        # Use explicit RK step-size controls
        'background_evolver': 'rk',
        'evolver': 'rk',
        'background_integration_stepsize': bg_stepsize,
        'perturbations_integration_stepsize': pert_stepsize,
        'perturbations_sampling_stepsize': pert_sampling_stepsize,

        # Tighten tolerances while testing convergence
        'tol_background_integration': tol_background,
        'tol_perturbations_integration': tol_pert,
    }

    M = Class()
    M.set(cosmofid)
    M.compute()
    background = M.get_background()
    return background, M


In [ ]:
# Fiducial full-field run
background_axion_1, M1 = compute_axion_nofluid(1e8, 0.4, 0.05)
background_z_1 = background_axion_1['z']

# Vary the mass
background_axion_2, M2 = compute_axion_nofluid(1e10, 0.4, 0.05)
background_axion_3, M3 = compute_axion_nofluid(1e5, 0.4, 0.05)

# Vary the density
background_axion_4, M4 = compute_axion_nofluid(1e8, 0.4, 0.15)
background_axion_5, M5 = compute_axion_nofluid(1e8, 0.4, 0.3)

# Vary f_a
background_axion_6, M6 = compute_axion_nofluid(1e8, 0.1, 0.05)
background_axion_7, M7 = compute_axion_nofluid(1e8, 0.6, 0.05)


In [ ]:
# Simple convergence test in the integrator step sizes
test_steps = [0.2, 0.1, 0.05, 0.02]
runs = {}

for step in test_steps:
    bg, cosmo = compute_axion_nofluid(1e8, 0.4, 0.05,\n                                     bg_stepsize=step,\n                                     pert_stepsize=step,\n                                     pert_sampling_stepsize=min(step, 0.05))
    runs[step] = (bg, cosmo)
    print(f'Completed run with background/perturbation step = {step}')


In [ ]:
# Inspect available background columns before plotting specific quantities
print(background_axion_1.dtype.names if hasattr(background_axion_1, 'dtype') else background_axion_1.keys())
